# Embedding Expansion and Dependency-based Aspect Extraction

This notebook evaluates two additional strategies for aspect-candidate extraction: semantic expansion with pretrained Word2Vec/FastText embeddings and dependency-based extraction with a Russian SynTagRus UDPipe model.


## 1. Setup


In [ ]:
!pip install gensim
!pip install wget
!pip install ufal.udpipe
!pip install --upgrade git+https://github.com/udapi/udapi-python.git

import itertools
import os
import re
from collections import Counter
from io import StringIO

import gensim
import wget
from udapi.block.read.conllu import Conllu
from ufal.udpipe import Model as UDModel
from ufal.udpipe import Pipeline


## 2. RusVectōrēs-compatible UDPipe preprocessing

The pretrained Word2Vec model expects lemmatized tokens with POS tags. The preprocessing below applies UDPipe-based normalization, including named-entity handling and optional POS retention.


In [ ]:
def replace_numbers(word):
    """Replace a numeric token with x characters of the same length."""
    return "x" * len(word)


def clean_token(token, misc):
    """Clean a UDPipe token."""
    processed_token = token.strip().replace(" ", "")
    if token == "Файл" and "SpaceAfter=No" in misc:
        return None
    return processed_token


def clean_lemma(lemma, pos):
    """Clean a UDPipe lemma."""
    processed_lemma = (
        lemma.strip()
        .replace(" ", "")
        .replace("_", "")
        .lower()
    )

    if (
        "|" in processed_lemma
        or processed_lemma.endswith(".jpg")
        or processed_lemma.endswith(".png")
    ):
        return None

    if pos != "PUNCT":
        if processed_lemma.startswith(("«", "»")):
            processed_lemma = processed_lemma[1:]
        if processed_lemma.endswith(("«", "»")):
            processed_lemma = processed_lemma[:-1]
        if processed_lemma.endswith(("!", "?", ",", ".")):
            processed_lemma = processed_lemma[:-1]

    return processed_lemma


def replace_symbols(search, replacement, text):
    """Replace every symbol from `search` that occurs in `text`."""
    matching_symbols = [
        symbol
        for symbol in search
        if symbol in text
    ]

    for symbol in matching_symbols:
        text = text.replace(
            symbol,
            replacement,
        )

    return text


def normalize_symbols(text):
    """Normalize Unicode symbols as in the original RusVectōrēs preprocessing script."""
    text = replace_symbols(
        "«»‹›„‚“‟‘‛”’",
        '"',
        text,
    )
    text = replace_symbols(
        "‒–—―‾̅¯",
        " -- ",
        text,
    )
    text = replace_symbols(
        "‐‑",
        "-",
        text,
    )
    text = replace_symbols(
        "         ​  ⁠　",
        " ",
        text,
    )

    text = re.sub("  ", " ", text)
    text = re.sub("\\t\\t", "\\t", text)

    text = replace_symbols(
        "ˌ̣̇•‣⁃⁌⁍∙◦·×⋅∙⁢",
        ".",
        text,
    )
    text = replace_symbols("∗", "*", text)
    text = replace_symbols("…", "...", text)
    text = replace_symbols("≁≋ⸯ҃", "∽", text)

    latin_replacements = {
        "Ä": "A",
        "ä": "a",
        "Ë": "E",
        "ë": "e",
        "Ḧ": "H",
        "ḧ": "h",
        "Ï": "I",
        "ï": "i",
        "Ö": "O",
        "ö": "o",
        "Ü": "U",
        "ü": "u",
        "Ÿ": "Y",
        "ÿ": "y",
        "ß": "s",
        "ẞ": "S",
    }

    for source_symbol, replacement in (
        latin_replacements.items()
    ):
        text = replace_symbols(
            source_symbol,
            replacement,
            text,
        )

    currencies = list(
        "₽$£₤€₪ℳ₾¢֏௹₼₡₠₴₧₰₿₣؋฿₩₴₲ƒ₫¥₭₡₺₦₱﷼៛₹₨₵৳₸₮ƒ"
    )

    alphabet = list(
        "\\t\\n\\r "
        "абвгдеёзжийклмнопрстуфхцчшщьыъэюя"
        "АБВГДЕЁЗЖИЙКЛМНОПРСТУФХЦЧШЩЬЫЪЭЮЯ"
        ",.[]{}()=+-−*&^%$#@!?~;:0123456789§/|"
        '"abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ '
    )
    alphabet.append("'")

    allowed_symbols = set(
        currencies + alphabet
    )

    return "".join(
        symbol
        for symbol in text
        if symbol in allowed_symbols
    )


def preprocess_for_rusvectores(
    pipeline,
    text="Строка",
    keep_pos=False,
    keep_punct=False,
):
    """Preprocess text with UDPipe for RusVectōrēs-compatible embeddings."""
    entities = {"PROPN"}
    named_entity_open = False
    proper_noun_memory = []
    memory_case = None
    memory_number = None
    tagged_tokens = []

    processed = pipeline.process(text)

    content = [
        line
        for line in processed.split("\\n")
        if not line.startswith("#")
    ]
    tagged = [
        line.split("\\t")
        for line in content
        if line
    ]

    for token_data in tagged:
        if len(token_data) != 10:
            continue

        (
            word_id,
            token,
            lemma,
            pos,
            xpos,
            feats,
            head,
            deprel,
            deps,
            misc,
        ) = token_data

        token = clean_token(
            token,
            misc,
        )
        lemma = clean_lemma(
            lemma,
            pos,
        )

        if not lemma or not token:
            continue

        if pos in entities:
            if "|" not in feats:
                tagged_tokens.append(
                    f"{lemma}_{pos}"
                )
                continue

            morphology = {
                feature.split("=")[0]:
                feature.split("=")[1]
                for feature in feats.split("|")
            }

            if (
                "Case" not in morphology
                or "Number" not in morphology
            ):
                tagged_tokens.append(
                    f"{lemma}_{pos}"
                )
                continue

            if not named_entity_open:
                named_entity_open = True
                memory_case = morphology["Case"]
                memory_number = morphology["Number"]

            if (
                morphology["Case"] == memory_case
                and morphology["Number"] == memory_number
            ):
                proper_noun_memory.append(
                    lemma
                )

                if (
                    "SpacesAfter=\\n" in misc
                    or "SpacesAfter=\\s\\n" in misc
                ):
                    named_entity_open = False
                    previous_lemma = "::".join(
                        proper_noun_memory
                    )
                    proper_noun_memory = []
                    tagged_tokens.append(
                        previous_lemma
                        + "_PROPN"
                    )
            else:
                named_entity_open = False
                previous_lemma = "::".join(
                    proper_noun_memory
                )
                proper_noun_memory = []
                tagged_tokens.append(
                    previous_lemma
                    + "_PROPN"
                )
                tagged_tokens.append(
                    f"{lemma}_{pos}"
                )

        else:
            if not named_entity_open:
                if (
                    pos == "NUM"
                    and token.isdigit()
                ):
                    lemma = replace_numbers(
                        token
                    )

                tagged_tokens.append(
                    f"{lemma}_{pos}"
                )

            else:
                named_entity_open = False
                previous_lemma = "::".join(
                    proper_noun_memory
                )
                proper_noun_memory = []
                tagged_tokens.append(
                    previous_lemma
                    + "_PROPN"
                )
                tagged_tokens.append(
                    f"{lemma}_{pos}"
                )

    if not keep_punct:
        tagged_tokens = [
            word
            for word in tagged_tokens
            if word.split("_")[1] != "PUNCT"
        ]

    if not keep_pos:
        tagged_tokens = [
            word.split("_")[0]
            for word in tagged_tokens
        ]

    return tagged_tokens


### Model paths

Set the paths below to the local pretrained Word2Vec, FastText, and UDPipe model files.


In [ ]:
WORD2VEC_MODEL_PATH = "/content/drive/MyDrive/model.bin"
FASTTEXT_MODEL_PATH = "/content/drive/MyDrive/fasttext 213/model.model"
UDPIPE_MODEL_PATH = (
    "/content/drive/MyDrive/udpipe/"
    "russian-syntagrus-ud-2.5-191206.udpipe"
)

word2vec_model = gensim.models.KeyedVectors.load_word2vec_format(
    WORD2VEC_MODEL_PATH,
    binary=True,
)
fasttext_model = gensim.models.KeyedVectors.load(FASTTEXT_MODEL_PATH)

rusvectores_ud_model = UDModel.load(UDPIPE_MODEL_PATH)
rusvectores_pipeline = Pipeline(
    rusvectores_ud_model,
    "tokenize",
    Pipeline.DEFAULT,
    Pipeline.DEFAULT,
    "conllu",
)


## 3. Prepare noun and aspect vocabularies

Noun and aspect candidates are converted to the POS-tagged representation expected by the Word2Vec model.


In [ ]:
def extract_nouns(tokenized_sentences):
    """Extract noun lemmas from spaCy token/POS pairs."""
    nouns = []

    for sentence in tokenized_sentences:
        for lemma, pos in sentence:
            if pos == "NOUN" and lemma != "-":
                nouns.append(lemma)

    return nouns


def prepare_pos_tagged_words(words, pipeline):
    """Convert single-word candidates to RusVectōrēs lemma_POS format."""
    tagged_words = []

    for word in words:
        if " " not in word:
            tagged_words.extend(
                preprocess_for_rusvectores(
                    pipeline,
                    word,
                    keep_pos=True,
                )
            )

    return tagged_words


In [ ]:
train_nouns = extract_nouns(train_tokens_and_labels)
test_nouns = extract_nouns(test_tokens_and_labels)
large_train_nouns = extract_nouns(large_train_tokens_and_labels)

train_nouns_pos = prepare_pos_tagged_words(
    train_nouns,
    rusvectores_pipeline,
)
test_nouns_pos = prepare_pos_tagged_words(
    test_nouns,
    rusvectores_pipeline,
)
large_train_nouns_pos = prepare_pos_tagged_words(
    large_train_nouns,
    rusvectores_pipeline,
)

train_aspects_pos = prepare_pos_tagged_words(
    train_supported["aspects"],
    rusvectores_pipeline,
)
test_aspects_pos = prepare_pos_tagged_words(
    test_supported["aspects"],
    rusvectores_pipeline,
)
large_train_aspects_pos = prepare_pos_tagged_words(
    large_train_supported["aspects"],
    rusvectores_pipeline,
)


## 4. Embedding-based aspect expansion


In [ ]:
SIMILARITY_THRESHOLDS = (
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
)


def expand_aspects_by_similarity(
    seed_aspects,
    candidate_words,
    excluded_words,
    embedding_model,
    thresholds=SIMILARITY_THRESHOLDS,
    require_noun_tag=False,
):
    """Expand aspect candidates at the similarity thresholds used in the thesis."""
    expansions = {threshold: [] for threshold in thresholds}

    for seed_aspect in seed_aspects:
        if seed_aspect not in embedding_model:
            continue

        for candidate_word in candidate_words:
            if (
                candidate_word not in embedding_model
                or candidate_word == seed_aspect
                or candidate_word in excluded_words
            ):
                continue

            similarity = embedding_model.similarity(
                seed_aspect,
                candidate_word,
            )

            for threshold in thresholds:
                if similarity > threshold:
                    expansions[threshold].append(candidate_word)

    if require_noun_tag:
        noun_expansions = {}

        for threshold, words in expansions.items():
            noun_expansions[threshold] = [
                word.rsplit("_", 1)[0]
                for word in words
                if word.rsplit("_", 1)[-1] == "NOUN"
            ]

        return noun_expansions

    return expansions


In [ ]:
word2vec_expansions = {
    "test": expand_aspects_by_similarity(
        test_aspects_pos,
        test_nouns_pos,
        test_aspects_pos,
        word2vec_model,
        require_noun_tag=True,
    ),
    "train": expand_aspects_by_similarity(
        test_aspects_pos,
        train_nouns_pos,
        train_aspects_pos,
        word2vec_model,
        require_noun_tag=True,
    ),
    "large_train": expand_aspects_by_similarity(
        large_train_aspects_pos,
        large_train_nouns_pos,
        large_train_aspects_pos,
        word2vec_model,
        require_noun_tag=True,
    ),
}

fasttext_expansions = {
    "test": expand_aspects_by_similarity(
        test_supported["aspects"],
        test_nouns,
        clean_propagated_aspects["test"],
        fasttext_model,
    ),
    "train": expand_aspects_by_similarity(
        train_supported["aspects"],
        train_nouns,
        clean_propagated_aspects["train"],
        fasttext_model,
    ),
    "large_train": expand_aspects_by_similarity(
        large_train_supported["aspects"],
        large_train_nouns,
        clean_propagated_aspects["large_train"],
        fasttext_model,
    ),
}


In [ ]:
def evaluate_expansion_thresholds(
    base_aspects,
    expansions,
    gold_aspects,
):
    """Evaluate every similarity threshold together with the base aspect set."""
    return {
        threshold: evaluate_aspects(
            base_aspects + expanded_aspects,
            gold_aspects,
        )
        for threshold, expanded_aspects in expansions.items()
    }


word2vec_test_scores = evaluate_expansion_thresholds(
    clean_propagated_aspects["test"],
    word2vec_expansions["test"],
    test_gold_lemmas,
)

fasttext_test_scores = evaluate_expansion_thresholds(
    clean_propagated_aspects["test"],
    fasttext_expansions["test"],
    test_gold_lemmas,
)

word2vec_test_scores, fasttext_test_scores


## 5. Dependency-based aspect extraction

The dependency parser extracts noun candidates linked to evaluative adjectives in the Russian SynTagRus dependency structure.


In [ ]:
dependency_ud_model = UDModel.load(UDPIPE_MODEL_PATH)
dependency_pipeline = Pipeline(
    dependency_ud_model,
    "tokenize",
    Pipeline.DEFAULT,
    Pipeline.DEFAULT,
    "conllu",
)


def extract_dependency_aspects(
    sentences,
    sentiment_adjectives,
):
    """Extract noun aspects linked to sentiment adjectives in dependency trees."""
    adjective_noun_pairs = []
    aspects = []

    for sentence in sentences:
        parsed_sentence = dependency_pipeline.process(sentence)
        tree = Conllu(
            filehandle=StringIO(parsed_sentence)
        ).read_tree()

        for node in tree.descendants:
            if (
                node.lemma in sentiment_adjectives
                and node.deprel == "root"
            ):
                for child in node.children:
                    if child.upos == "NOUN":
                        adjective_noun_pairs.append(
                            f"{node.lemma} {child.lemma}"
                        )
                        aspects.append(child.lemma.lower())

            elif (
                node.lemma in sentiment_adjectives
                and node.deprel != "root"
                and node.parent.upos == "NOUN"
            ):
                adjective_noun_pairs.append(
                    f"{node.lemma} {node.parent.lemma}"
                )
                aspects.append(node.parent.lemma.lower())

    return adjective_noun_pairs, aspects


In [ ]:
dependency_results = {}

for dataset_name, sentences in {
    "train": train_texts,
    "test": test_texts,
    "large_train": large_train_texts,
}.items():
    pairs, aspects = extract_dependency_aspects(
        sentences,
        rusentilex_adjectives,
    )
    dependency_results[dataset_name] = {
        "pairs": pairs,
        "aspects": aspects,
    }


In [ ]:
dependency_test_scores = evaluate_aspects(
    dependency_results["test"]["aspects"],
    test_gold_lemmas,
)

train_dependency_overlap = list(
    (
        Counter(test_gold_lemmas)
        & Counter(dependency_results["train"]["aspects"])
    ).elements()
)

large_train_dependency_overlap = list(
    (
        Counter(test_gold_lemmas)
        & Counter(dependency_results["large_train"]["aspects"])
    ).elements()
)

combined_dependency_scores = evaluate_aspects(
    (
        dependency_results["test"]["aspects"]
        + train_dependency_overlap
        + large_train_dependency_overlap
    ),
    test_gold_lemmas,
)

dependency_test_scores, combined_dependency_scores
